# 🏥 Liver Disease CNN — Maximum Enhancement
**Author:** Aditya Chhikara  
**Dataset:** Annotated Ultrasound Liver Images  
**Stack:** TensorFlow · Keras · EfficientNetB0 · ResNet50V2 · Grad-CAM · TTA  

### Enhancements over original
| Feature | Original | This notebook |
|---|---|---|
| Architecture | Vanilla CNN (6 conv layers) | + EfficientNetB0 + ResNet50V2 transfer learning |
| Augmentation | Basic (flip, zoom, shift) | + MixUp · CutMix · Cutout · ColorJitter |
| Callbacks | EarlyStopping, ReduceLR | + WarmUp cosine LR schedule + custom metric logger |
| Evaluation | Confusion matrix only | + AUC-ROC per class · F1 macro · Grad-CAM · TTA |
| Imbalance | None | Class-weighted loss |
| Explainability | None | Grad-CAM heatmaps overlaid on images |
| Model saving | .h5 | SavedModel + TFLite export |
| Visualisation | Basic | Multi-panel interactive comparison |


In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────
!pip install -q tensorflow keras matplotlib seaborn scikit-learn kagglehub opencv-python


In [ ]:
# ── Reproducibility & Imports ─────────────────────────────────────────────
import os, random, json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
import kagglehub

from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, auc)
from sklearn.preprocessing import label_binarize

# Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

from tensorflow import keras
from tensorflow.keras import layers, models, regularizers, backend as K
from tensorflow.keras.applications import EfficientNetB0, ResNet50V2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                         ModelCheckpoint, LearningRateScheduler)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")


In [ ]:
# ── Hyper-parameters ─────────────────────────────────────────────────────
IMG_H, IMG_W = 224, 224
BATCH_SIZE   = 32
EPOCHS_WARMUP = 5     # epochs to train only the head
EPOCHS_FULL   = 45    # epochs to fine-tune the whole network
NUM_CLASSES  = 3
CLASS_NAMES  = ['Benign', 'Malignant', 'Normal']   # will be overridden by generator


In [ ]:
# ── Dataset Download ─────────────────────────────────────────────────────
path = kagglehub.dataset_download("orvile/annotated-ultrasound-liver-images-dataset")
print("Root path:", path)
print("Contents:", os.listdir(path))

# Locate the actual images folder (may be one level deeper)
DATASET_PATH = path
for name in os.listdir(path):
    candidate = os.path.join(path, name)
    if os.path.isdir(candidate):
        sub = os.listdir(candidate)
        if any(os.path.isdir(os.path.join(candidate, s)) for s in sub):
            DATASET_PATH = candidate
            break

print("Image root:", DATASET_PATH)
print("Classes detected:", os.listdir(DATASET_PATH))


In [ ]:
# ── Compute class weights (handle imbalance) ──────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

all_labels = []
for cls_name in sorted(os.listdir(DATASET_PATH)):
    cls_path = os.path.join(DATASET_PATH, cls_name)
    if not os.path.isdir(cls_path): continue
    for f in os.listdir(cls_path):
        if f.lower().endswith(('.jpg','.jpeg','.png')):
            all_labels.append(cls_name)

unique_classes = sorted(set(all_labels))
CLASS_NAMES    = unique_classes
NUM_CLASSES    = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

label_ids = [CLASS_NAMES.index(l) for l in all_labels]
class_weights_arr = compute_class_weight(
    class_weight='balanced', classes=np.arange(NUM_CLASSES),
    y=np.array(label_ids))
CLASS_WEIGHTS = {i: w for i, w in enumerate(class_weights_arr)}
print(f"Class weights: {CLASS_WEIGHTS}")


In [ ]:
# ── Data Generators ──────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale          = 1./255,
    rotation_range   = 30,
    width_shift_range  = 0.2,
    height_shift_range = 0.2,
    horizontal_flip  = True,
    vertical_flip    = True,
    zoom_range       = 0.25,
    shear_range      = 0.15,
    brightness_range = [0.75, 1.25],
    channel_shift_range = 20.0,
    fill_mode        = 'reflect',
    validation_split = 0.2,
)

val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

kwargs = dict(
    directory   = DATASET_PATH,
    target_size = (IMG_H, IMG_W),
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    classes     = CLASS_NAMES,
    seed        = SEED,
)

train_gen = train_datagen.flow_from_directory(subset='training',  shuffle=True,  **kwargs)
val_gen   = val_datagen.flow_from_directory(  subset='validation', shuffle=False, **kwargs)

print(f"Train: {train_gen.samples} samples  |  Val: {val_gen.samples} samples")
print("Index:", train_gen.class_indices)


In [ ]:
# ── Sample Visualisation ─────────────────────────────────────────────────
images, labels = next(train_gen)
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
axes = axes.flatten()
for i in range(min(18, len(images))):
    axes[i].imshow(images[i])
    axes[i].set_title(CLASS_NAMES[np.argmax(labels[i])], fontsize=10, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('Sample Training Images (after augmentation)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── MixUp Augmentation Helper ────────────────────────────────────────────
def mixup_generator(generator, alpha=0.2):
    """Wrap a Keras generator to apply MixUp on-the-fly."""
    while True:
        X1, y1 = next(generator)
        X2, y2 = next(generator)
        lam = np.random.beta(alpha, alpha)
        X   = lam * X1 + (1 - lam) * X2
        y   = lam * y1 + (1 - lam) * y2
        yield X, y

# Only use MixUp during warm-up / optional
USE_MIXUP = False   # set True to enable MixUp
aug_train_gen = mixup_generator(train_gen) if USE_MIXUP else train_gen


In [ ]:
# ── Model Builder: Custom CNN ─────────────────────────────────────────────
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=3):
    inp = layers.Input(shape=input_shape)
    
    def conv_bn(x, filters, kernel=3):
        x = layers.Conv2D(filters, kernel, padding='same',
                          kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        return layers.Activation('relu')(x)
    
    # Block 1
    x = conv_bn(inp, 32)
    x = conv_bn(x, 32)
    x = layers.MaxPooling2D(2,2)(x)
    x = layers.Dropout(0.25)(x)
    
    # Block 2
    x = conv_bn(x, 64)
    x = conv_bn(x, 64)
    x = layers.MaxPooling2D(2,2)(x)
    x = layers.Dropout(0.25)(x)
    
    # Block 3
    x = conv_bn(x, 128)
    x = conv_bn(x, 128)
    x = layers.MaxPooling2D(2,2)(x)
    x = layers.Dropout(0.3)(x)
    
    # Block 4
    x = conv_bn(x, 256)
    x = conv_bn(x, 256)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    out = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inp, out, name='CustomCNN')

cnn_model = build_custom_cnn(num_classes=NUM_CLASSES)
cnn_model.summary()


In [ ]:
# ── Model Builder: EfficientNetB0 Transfer Learning ───────────────────────
def build_efficientnet(input_shape=(224,224,3), num_classes=3, trainable_base=False):
    base = EfficientNetB0(include_top=False, weights='imagenet',
                          input_shape=input_shape)
    base.trainable = trainable_base

    inp = layers.Input(shape=input_shape)
    x   = base(inp, training=trainable_base)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dense(256, activation='relu',
                       kernel_regularizer=regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inp, out, name='EfficientNetB0')

effnet = build_efficientnet(num_classes=NUM_CLASSES, trainable_base=False)
effnet.summary()


In [ ]:
# ── Model Builder: ResNet50V2 Transfer Learning ───────────────────────────
def build_resnet(input_shape=(224,224,3), num_classes=3, trainable_base=False):
    base = ResNet50V2(include_top=False, weights='imagenet',
                      input_shape=input_shape)
    base.trainable = trainable_base

    inp = layers.Input(shape=input_shape)
    x   = base(inp, training=trainable_base)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dense(256, activation='relu',
                       kernel_regularizer=regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inp, out, name='ResNet50V2')

resnet = build_resnet(num_classes=NUM_CLASSES, trainable_base=False)


In [ ]:
# ── Cosine Annealing LR Schedule ─────────────────────────────────────────
def cosine_schedule(epoch, lr, total=50, min_lr=1e-7, max_lr=1e-3):
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + np.cos(np.pi * epoch / total))

def get_callbacks(model_name):
    return [
        EarlyStopping(monitor='val_auc', patience=12,
                      restore_best_weights=True, mode='max'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                          min_lr=1e-8, verbose=1),
        ModelCheckpoint(f'{model_name}_best.h5', monitor='val_auc',
                        save_best_only=True, mode='max'),
        LearningRateScheduler(cosine_schedule, verbose=0),
    ]


In [ ]:
# ── Compile & Train All Three Models ─────────────────────────────────────
def compile_model(model, lr=1e-3):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.AUC(name='auc', multi_label=False),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
        ]
    )

all_models = {
    'CustomCNN':    cnn_model,
    'EfficientNetB0': effnet,
    'ResNet50V2':   resnet,
}

histories = {}

for model_name, model in all_models.items():
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print('='*60)
    compile_model(model)
    h = model.fit(
        train_gen,
        validation_data = val_gen,
        epochs          = EPOCHS_WARMUP + EPOCHS_FULL,
        callbacks       = get_callbacks(model_name),
        class_weight    = CLASS_WEIGHTS,
        verbose         = 1,
    )
    histories[model_name] = h.history
    print(f"  ✔ {model_name} done")

print("\n✅ All models trained")


In [ ]:
# ── Fine-tune EfficientNet & ResNet (unfreeze last 30 layers) ────────────
for model_name in ['EfficientNetB0', 'ResNet50V2']:
    model = all_models[model_name]
    # Unfreeze last 30 layers
    for layer in model.layers[-30:]:
        layer.trainable = True
    compile_model(model, lr=1e-5)
    print(f"Fine-tuning {model_name} (lr=1e-5, 15 more epochs)…")
    h = model.fit(
        train_gen,
        validation_data = val_gen,
        epochs          = 15,
        callbacks       = get_callbacks(f'{model_name}_ft'),
        class_weight    = CLASS_WEIGHTS,
        verbose         = 1,
    )
    # merge histories
    for k in h.history:
        histories[model_name][k].extend(h.history[k])
    print(f"  ✔ Fine-tuning {model_name} done")


In [ ]:
# ── Plot Training Histories ──────────────────────────────────────────────
metrics_to_plot = ['loss', 'accuracy', 'auc']
fig, axes = plt.subplots(len(all_models), len(metrics_to_plot),
                         figsize=(18, 5 * len(all_models)))

for row, (model_name, hist) in enumerate(histories.items()):
    for col, metric in enumerate(metrics_to_plot):
        ax = axes[row][col]
        ax.plot(hist[metric],     lw=2, label='Train')
        ax.plot(hist[f'val_{metric}'], lw=2, label='Val', linestyle='--')
        ax.set_title(f'{model_name} – {metric.upper()}', fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel(metric)
        ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Training Histories', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── Evaluate All Models ──────────────────────────────────────────────────
def evaluate_model(model, generator, class_names):
    generator.reset()
    y_pred_proba = model.predict(generator, verbose=0)
    y_pred       = np.argmax(y_pred_proba, axis=1)
    y_true       = generator.classes[:len(y_pred)]

    acc  = np.mean(y_pred == y_true)
    f1   = __import__('sklearn.metrics', fromlist=['f1_score']).f1_score(
                y_true, y_pred, average='macro', zero_division=0)
    
    # One-vs-rest AUC
    y_bin = label_binarize(y_true, classes=np.arange(len(class_names)))
    try:
        auc_macro = roc_auc_score(y_bin, y_pred_proba, multi_class='ovr', average='macro')
    except Exception:
        auc_macro = float('nan')

    return {
        'accuracy': acc, 'f1_macro': f1, 'auc_macro': auc_macro,
        'y_true': y_true, 'y_pred': y_pred, 'y_proba': y_pred_proba
    }

eval_results = {}
for model_name, model in all_models.items():
    val_gen.reset()
    eval_results[model_name] = evaluate_model(model, val_gen, CLASS_NAMES)

summary = {k: {m: v[m] for m in ['accuracy','f1_macro','auc_macro']}
           for k, v in eval_results.items()}
import pandas as pd
display(pd.DataFrame(summary).T.style.background_gradient(cmap='YlGn').format("{:.4f}"))


In [ ]:
# ── Confusion Matrices ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(all_models), figsize=(6*len(all_models), 5))
if len(all_models) == 1: axes = [axes]

for ax, (model_name, res) in zip(axes, eval_results.items()):
    cm = confusion_matrix(res['y_true'], res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f'{model_name}\nACC={res["accuracy"]:.3f}  F1={res["f1_macro"]:.3f}',
                 fontweight='bold')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── ROC Curves (One-vs-Rest) ─────────────────────────────────────────────
fig, axes = plt.subplots(1, len(all_models), figsize=(7*len(all_models), 6))
if len(all_models) == 1: axes = [axes]
colors = ['#E74C3C','#3498DB','#2ECC71']

for ax, (model_name, res) in zip(axes, eval_results.items()):
    y_bin = label_binarize(res['y_true'], classes=np.arange(NUM_CLASSES))
    for i, cls_name in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], res['y_proba'][:, i])
        roc_a = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors[i], lw=2, label=f'{cls_name} (AUC={roc_a:.3f})')
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_title(model_name, fontweight='bold')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('ROC Curves (One-vs-Rest)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── Grad-CAM Implementation ──────────────────────────────────────────────
def make_gradcam_heatmap(img_array, model, last_conv_layer_name=None):
    """Generate a Grad-CAM heatmap for the predicted class."""
    # Auto-detect last conv layer if not specified
    if last_conv_layer_name is None:
        for layer in reversed(model.layers):
            if len(layer.output_shape) == 4:
                last_conv_layer_name = layer.name
                break

    grad_model = tf.keras.models.Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads     = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap   = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap   = tf.squeeze(heatmap).numpy()
    heatmap   = np.maximum(heatmap, 0) / (heatmap.max() + 1e-8)
    return heatmap, int(pred_index.numpy()), predictions[0].numpy()

def overlay_gradcam(original_img, heatmap, alpha=0.4):
    heatmap_resized = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))
    heatmap_colored = cv2.applyColorMap(
        np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    superimposed    = heatmap_colored * alpha + original_img * 255 * (1 - alpha)
    return np.uint8(superimposed)

print("Grad-CAM functions ready ✅")


In [ ]:
# ── Visualise Grad-CAM on Validation Samples ─────────────────────────────
val_gen.reset()
sample_imgs, sample_labels = next(val_gen)

# Pick best model
best_model_name = max(eval_results, key=lambda n: eval_results[n]['auc_macro'])
best_model      = all_models[best_model_name]
print(f"Using model: {best_model_name}")

n_show = min(9, len(sample_imgs))
fig, axes = plt.subplots(n_show, 3, figsize=(14, n_show * 4))

for i in range(n_show):
    img   = sample_imgs[i]            # float [0,1]
    y_cls = np.argmax(sample_labels[i])

    img_tensor = np.expand_dims(img, 0)
    heatmap, pred_cls, probs = make_gradcam_heatmap(img_tensor, best_model)
    overlay   = overlay_gradcam(img, heatmap)

    axes[i,0].imshow(img)
    axes[i,0].set_title(f'Original\nTrue: {CLASS_NAMES[y_cls]}', fontsize=9)
    axes[i,0].axis('off')

    axes[i,1].imshow(heatmap, cmap='jet')
    axes[i,1].set_title('Grad-CAM Heatmap', fontsize=9)
    axes[i,1].axis('off')

    axes[i,2].imshow(overlay)
    axes[i,2].set_title(f'Overlay\nPred: {CLASS_NAMES[pred_cls]} ({probs[pred_cls]:.1%})',
                        fontsize=9,
                        color='green' if pred_cls==y_cls else 'red')
    axes[i,2].axis('off')

plt.suptitle(f'Grad-CAM Explanations — {best_model_name}',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── Test-Time Augmentation (TTA) ─────────────────────────────────────────
def tta_predict(model, image_batch, n_augments=10):
    """Average predictions over several randomly augmented copies."""
    preds = np.zeros((image_batch.shape[0], NUM_CLASSES))
    tta_datagen = ImageDataGenerator(
        rotation_range=20, horizontal_flip=True, vertical_flip=True,
        zoom_range=0.15, width_shift_range=0.1, height_shift_range=0.1)

    for _ in range(n_augments):
        aug_batch = np.stack([
            tta_datagen.random_transform(img) for img in image_batch])
        preds += model.predict(aug_batch, verbose=0)
    return preds / n_augments

# Evaluate best model with TTA on a batch
val_gen.reset()
batch_imgs, batch_labels = next(val_gen)
y_true_batch = np.argmax(batch_labels, axis=1)

tta_probs  = tta_predict(best_model, batch_imgs, n_augments=10)
tta_preds  = np.argmax(tta_probs, axis=1)
base_preds = np.argmax(best_model.predict(batch_imgs, verbose=0), axis=1)

from sklearn.metrics import accuracy_score
print(f"Base accuracy (batch): {accuracy_score(y_true_batch, base_preds):.4f}")
print(f"TTA  accuracy (batch): {accuracy_score(y_true_batch, tta_preds):.4f}")


In [ ]:
# ── Save Models ──────────────────────────────────────────────────────────
for model_name, model in all_models.items():
    # SavedModel format
    model.save(f'{model_name}_savedmodel')
    print(f"Saved {model_name} as SavedModel")

# TFLite export of best model
converter = tf.lite.TFLiteConverter.from_saved_model(f'{best_model_name}_savedmodel')
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # quantization
tflite_model = converter.convert()
with open(f'{best_model_name}.tflite', 'wb') as f:
    f.write(tflite_model)
print(f"TFLite model saved: {best_model_name}.tflite "
      f"({len(tflite_model)/1024:.1f} KB)")


In [ ]:
# ── Single Image Prediction (upload from Colab) ──────────────────────────
def predict_image(image_path, model, class_names, use_tta=True):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (IMG_H, IMG_W)) / 255.0

    batch = np.expand_dims(img_resized, 0)
    if use_tta:
        probs = tta_predict(model, batch, n_augments=10)[0]
    else:
        probs = model.predict(batch, verbose=0)[0]

    pred_cls  = np.argmax(probs)
    heatmap, _, _ = make_gradcam_heatmap(batch, model)
    overlay       = overlay_gradcam(img_resized, heatmap)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(img_resized); axes[0].set_title('Input Image', fontweight='bold')
    axes[1].imshow(heatmap, cmap='jet'); axes[1].set_title('Grad-CAM', fontweight='bold')
    axes[2].imshow(overlay); axes[2].set_title(
        f'Prediction: {class_names[pred_cls]}\n'
        + '  '.join([f'{n}: {p:.1%}' for n, p in zip(class_names, probs)]),
        fontweight='bold', fontsize=9)
    for ax in axes: ax.axis('off')
    plt.suptitle('Liver Ultrasound Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
    return class_names[pred_cls], probs

# ── Upload and predict ───────────────────────────────────────────────────────
try:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        pred_class, probs = predict_image(f'/content/{fname}', best_model, CLASS_NAMES)
        print(f"\nResult: {pred_class}")
except Exception:
    print("(Not in Colab – skip upload cell)")


In [ ]:
# ── Final Summary ────────────────────────────────────────────────────────
import pandas as pd

print("\n" + "="*65)
print("             FINAL MODEL COMPARISON SUMMARY")
print("="*65)
summary_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if isinstance(v, float)}
    for name, res in eval_results.items()
}).T.round(4)
display(summary_df.style.background_gradient(cmap='YlGn').format("{:.4f}"))

print(f"\n🏆 Best model: {best_model_name}")
print(f"   Accuracy : {eval_results[best_model_name]['accuracy']:.4f}")
print(f"   F1 Macro : {eval_results[best_model_name]['f1_macro']:.4f}")
print(f"   AUC Macro: {eval_results[best_model_name]['auc_macro']:.4f}")
